In [ ]:
# Blood Cell Detection - Comparing Loss Functions
# Run this notebook from the repository root.
%pip install -r requirements-notebook.txt

import os
import shutil
import zipfile
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from ultralytics import YOLO


In [ ]:
from pathlib import Path

# Reuse the included dataset; no upload is required after cloning.
data_yaml_path = Path('blood_cell_dataset/data.yaml')
if not data_yaml_path.is_file():
    raise FileNotFoundError('Run the notebook from the repository root with blood_cell_dataset present.')
data_yaml_path = str(data_yaml_path.resolve())
print(f'Dataset configuration: {data_yaml_path}')


In [ ]:
# Keep the checked-in YAML portable. Write absolute runtime paths separately.
with open(data_yaml_path, encoding='utf-8') as f:
    data_config = yaml.safe_load(f)
base_dir = os.path.dirname(os.path.abspath(data_yaml_path))
for split, folder in [('train', 'train'), ('val', 'valid'), ('test', 'test')]:
    data_config[split] = os.path.join(base_dir, folder)
data_config['path'] = base_dir
runtime_dir = Path('outputs/notebook')
runtime_dir.mkdir(parents=True, exist_ok=True)
data_yaml_path = str((runtime_dir / 'data.resolved.yaml').resolve())
with open(data_yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_config, f, sort_keys=False)
for split in ['train', 'val', 'test']:
    images = Path(data_config[split]) / 'images'
    print(f'{split}: {len(list(images.glob("*.jpg")))} images')


In [ ]:
print("\n" + "="*70)
print("TRAINING CONFIGURATION")
print("="*70)

# Reduced parameters for Lightning AI (free tier)
MODEL_SIZE = 'yolov8n.pt'  # Nano - fastest
EPOCHS = 30  # Reduced from 50
IMG_SIZE = 640
BATCH_SIZE = 8  # Reduced from 16 to avoid memory issues

print(f"\nTraining Parameters:")
print(f"  Model: {MODEL_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Image Size: {IMG_SIZE}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"\nEstimated time per experiment: ~15-20 minutes")
print(f"Total experiments: 4 (Baseline + 3 loss variations)")
print(f"Total estimated time: ~1-1.5 hours")

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 1/4: BASELINE - Default CIoU Loss")
print("="*70)

model_baseline = YOLO(MODEL_SIZE)

results_baseline = model_baseline.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project='blood_cell_experiments',
    name='baseline_ciou',
    verbose=True,
    patience=10  # Early stopping
)

print("✅ Baseline training complete!")

# Save progress
all_results = {
    'Baseline (CIoU)': {
        'model_path': 'blood_cell_experiments/baseline_ciou/weights/best.pt',
        'config': 'Default settings'
    }
}

SECTION 6: Experiment 2 - High Box Loss Weight

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 2/4: High Box Loss Weight")
print("="*70)
print("Focus: Emphasize bounding box localization accuracy")

model_box = YOLO(MODEL_SIZE)

results_box = model_box.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    box=10.0,  # Increased from default 7.5
    project='blood_cell_experiments',
    name='high_box_loss',
    verbose=True,
    patience=10
)

all_results['High Box Loss'] = {
    'model_path': 'blood_cell_experiments/high_box_loss/weights/best.pt',
    'config': 'box=10.0'
}

print("✅ High Box Loss training complete!")

SECTION 7: Experiment 3 - High DFL Weight

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 3/4: High DFL (Distribution Focal Loss) Weight")
print("="*70)
print("Focus: Better handle small object detection (platelets)")

model_dfl = YOLO(MODEL_SIZE)

results_dfl = model_dfl.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    dfl=2.5,  # Increased from default 1.5
    project='blood_cell_experiments',
    name='high_dfl_loss',
    verbose=True,
    patience=10
)

all_results['High DFL Loss'] = {
    'model_path': 'blood_cell_experiments/high_dfl_loss/weights/best.pt',
    'config': 'dfl=2.5'
}

print("✅ High DFL Loss training complete!")

SECTION 8: Experiment 4 - Balanced Configuration

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 4/4: Balanced Loss Configuration")
print("="*70)
print("Focus: Balance between classification and localization")

model_balanced = YOLO(MODEL_SIZE)

results_balanced = model_balanced.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    cls=1.0,  # Increased from default 0.5
    box=9.0,  # Slightly increased
    dfl=2.0,  # Slightly increased
    project='blood_cell_experiments',
    name='balanced_loss',
    verbose=True,
    patience=10
)

all_results['Balanced Loss'] = {
    'model_path': 'blood_cell_experiments/balanced_loss/weights/best.pt',
    'config': 'cls=1.0, box=9.0, dfl=2.0'
}

print("✅ Balanced Loss training complete!")
print("\n🎉 ALL TRAINING COMPLETE!")

SECTION 9: Evaluation and Comparison

In [ ]:
print("\n" + "="*70)
print("EVALUATION AND COMPARISON")
print("="*70)

# Evaluate all models
comparison_data = []

for exp_name, info in all_results.items():
    print(f"\nEvaluating {exp_name}...")
    
    model = YOLO(info['model_path'])
    metrics = model.val(data=data_yaml_path, split='test')
    
    comparison_data.append({
        'Experiment': exp_name,
        'Configuration': info['config'],
        'mAP@0.5': round(metrics.box.map50, 4),
        'mAP@0.5:0.95': round(metrics.box.map, 4),
        'Precision': round(metrics.box.mp, 4),
        'Recall': round(metrics.box.mr, 4),
        'F1-Score': round(2 * (metrics.box.mp * metrics.box.mr) / (metrics.box.mp + metrics.box.mr + 1e-6), 4)
    })

# Create comparison DataFrame
comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('mAP@0.5:0.95', ascending=False)

print("\n" + "="*70)
print("RESULTS COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))

# Save results
comparison_df.to_csv('blood_cell_experiments/results_comparison.csv', index=False)
print("\n📁 Results saved to: blood_cell_experiments/results_comparison.csv")

SECTION 10: Visualizations

In [ ]:
# Create comparison visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# mAP@0.5
axes[0, 0].bar(comparison_df['Experiment'], comparison_df['mAP@0.5'], 
               color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
axes[0, 0].set_title('mAP@0.5 Comparison', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Score')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(comparison_df['mAP@0.5']):
    axes[0, 0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# mAP@0.5:0.95
axes[0, 1].bar(comparison_df['Experiment'], comparison_df['mAP@0.5:0.95'], 
               color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
axes[0, 1].set_title('mAP@0.5:0.95 Comparison', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Score')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(comparison_df['mAP@0.5:0.95']):
    axes[0, 1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# Precision
axes[1, 0].bar(comparison_df['Experiment'], comparison_df['Precision'], 
               color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
axes[1, 0].set_title('Precision Comparison', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Score')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(comparison_df['Precision']):
    axes[1, 0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# Recall
axes[1, 1].bar(comparison_df['Experiment'], comparison_df['Recall'], 
               color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
axes[1, 1].set_title('Recall Comparison', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Score')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(comparison_df['Recall']):
    axes[1, 1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

plt.suptitle('Blood Cell Detection: Loss Function Comparison', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('blood_cell_experiments/comparison_chart.png', dpi=300, bbox_inches='tight')
plt.show()

# Best model summary
best_model_row = comparison_df.iloc[0]
print("\n" + "="*70)
print(f"🏆 BEST MODEL: {best_model_row['Experiment']}")
print("="*70)
print(f"   Configuration: {best_model_row['Configuration']}")
print(f"   mAP@0.5: {best_model_row['mAP@0.5']}")
print(f"   mAP@0.5:0.95: {best_model_row['mAP@0.5:0.95']}")
print(f"   Precision: {best_model_row['Precision']}")
print(f"   Recall: {best_model_row['Recall']}")
print(f"   F1-Score: {best_model_row['F1-Score']}")

best_model_path = all_results[best_model_row['Experiment']]['model_path']

SECTION 11: Cell Counting Demo

In [ ]:
print("\n" + "="*70)
print("BLOOD CELL COUNTING WITH BEST MODEL")
print("="*70)

# Load best model
best_model = YOLO(best_model_path)

def count_and_visualize_cells(image_path, model, class_names):
    """Detect, count, and visualize blood cells"""
    results = model(image_path)[0]
    cell_counts = {name: 0 for name in class_names}
    
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    if len(results.boxes) > 0:
        for box in results.boxes:
            class_id = int(box.cls[0])
            class_name = class_names[class_id]
            cell_counts[class_name] += 1
            
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            
            colors = {'RBC': (255, 0, 0), 'WBC': (0, 255, 0), 'platelets': (0, 0, 255)}
            color = colors.get(class_name, (255, 255, 0))
            
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, f"{class_name} {conf:.2f}", (x1, y1-5), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    return img, cell_counts

# Test on sample images
test_path = os.path.join(data_config['test'], 'images')
test_images = [f for f in os.listdir(test_path) if f.endswith(('.jpg', '.png'))][:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, img_name in enumerate(test_images):
    img_path = os.path.join(test_path, img_name)
    annotated_img, counts = count_and_visualize_cells(img_path, best_model, data_config['names'])
    
    axes[idx].imshow(annotated_img)
    title = f"Image {idx+1}\n"
    for cell_type, count in counts.items():
        title += f"{cell_type}: {count}  "
    axes[idx].set_title(title, fontsize=9)
    axes[idx].axis('off')

plt.suptitle(f'Detection Results - {best_model_row["Experiment"]}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('blood_cell_experiments/detection_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ PROJECT COMPLETE!")

SECTION 12: Model Size Experiments

In [ ]:
print("\n" + "="*70)
print("ADDITIONAL EXPERIMENTS: MODEL SIZE COMPARISON")
print("="*70)

print("""
YOLOv8 Model Variants:
- YOLOv8n (Nano): 3.2M params, fastest
- YOLOv8s (Small): 11.2M params, balanced
- YOLOv8m (Medium): 25.9M params, most accurate

We'll compare these to see the speed-accuracy trade-off.
""")

# If you already have baseline with yolov8n, we'll add s and m
model_variants = ['yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt']
model_names = ['YOLOv8n', 'YOLOv8s', 'YOLOv8m']

# Store results
model_comparison = {}

for model_size, model_name in zip(model_variants, model_names):
    print(f"\n{'='*70}")
    print(f"Training with {model_name}")
    print(f"{'='*70}")
    
    # Adjust batch size for larger models
    if 'n' in model_size:
        batch = 8
        time_est = "15-20 min"
    elif 's' in model_size:
        batch = 8
        time_est = "20-30 min"
    else:  # medium
        batch = 4  # Reduce for memory
        time_est = "30-40 min"
    
    print(f"Estimated time: {time_est}")
    
    model = YOLO(model_size)
    results = model.train(
        data=data_yaml_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=batch,
        project='blood_cell_experiments',
        name=f'model_size_{model_name.lower()}',
        verbose=True,
        patience=10
    )
    
    model_comparison[model_name] = {
        'model_path': f'blood_cell_experiments/model_size_{model_name.lower()}/weights/best.pt',
        'batch_size': batch,
        'params': '3.2M' if 'n' in model_size else '11.2M' if 's' in model_size else '25.9M'
    }
    
    print(f"✅ {model_name} training complete!")

print("\n🎉 All model size experiments completed!")

SECTION 13: Comprehensive Model Size Evaluation

In [ ]:
print("\n" + "="*70)
print("MODEL SIZE COMPARISON EVALUATION")
print("="*70)

import time

model_eval_data = []

for model_name, info in model_comparison.items():
    print(f"\nEvaluating {model_name}...")
    
    model = YOLO(info['model_path'])
    
    # Evaluate accuracy
    metrics = model.val(data=data_yaml_path, split='test')
    
    # Measure inference speed
    test_img = os.path.join(data_config['test'], 'images', 
                           os.listdir(os.path.join(data_config['test'], 'images'))[0])
    
    # Warmup
    for _ in range(5):
        _ = model(test_img, verbose=False)
    
    # Time 10 inferences
    start = time.time()
    for _ in range(10):
        _ = model(test_img, verbose=False)
    avg_time = (time.time() - start) / 10
    fps = 1 / avg_time
    
    model_eval_data.append({
        'Model': model_name,
        'Parameters': info['params'],
        'mAP@0.5': round(metrics.box.map50, 4),
        'mAP@0.5:0.95': round(metrics.box.map, 4),
        'Precision': round(metrics.box.mp, 4),
        'Recall': round(metrics.box.mr, 4),
        'Inference Time (ms)': round(avg_time * 1000, 2),
        'FPS': round(fps, 2)
    })

# Create DataFrame
model_comparison_df = pd.DataFrame(model_eval_data)

print("\n" + "="*70)
print("MODEL SIZE COMPARISON RESULTS")
print("="*70)
print(model_comparison_df.to_string(index=False))

# Save results
model_comparison_df.to_csv('blood_cell_experiments/model_size_comparison.csv', index=False)
print("\n📁 Model comparison saved!")

SECTION 14: Advanced Visualizations - Model Comparison

In [ ]:
# Comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Accuracy vs Model Size
ax1 = axes[0, 0]
x = np.arange(len(model_comparison_df))
width = 0.35
ax1.bar(x - width/2, model_comparison_df['mAP@0.5'], width, label='mAP@0.5', alpha=0.8)
ax1.bar(x + width/2, model_comparison_df['mAP@0.5:0.95'], width, label='mAP@0.5:0.95', alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(model_comparison_df['Model'])
ax1.set_ylabel('Score')
ax1.set_title('Accuracy Comparison', fontweight='bold')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# 2. Speed vs Accuracy Trade-off
ax2 = axes[0, 1]
scatter = ax2.scatter(model_comparison_df['Inference Time (ms)'], 
                     model_comparison_df['mAP@0.5:0.95'],
                     s=300, alpha=0.6, c=['#FF6B6B', '#4ECDC4', '#45B7D1'])
ax2.set_xlabel('Inference Time (ms)', fontweight='bold')
ax2.set_ylabel('mAP@0.5:0.95', fontweight='bold')
ax2.set_title('Speed-Accuracy Trade-off', fontweight='bold')
ax2.grid(alpha=0.3)
for idx, row in model_comparison_df.iterrows():
    ax2.annotate(row['Model'], (row['Inference Time (ms)'], row['mAP@0.5:0.95']),
                fontsize=10, ha='center', xytext=(0, 10), textcoords='offset points')

# 3. FPS Comparison
ax3 = axes[0, 2]
bars = ax3.barh(model_comparison_df['Model'], model_comparison_df['FPS'], 
                color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
ax3.set_xlabel('FPS (Frames Per Second)', fontweight='bold')
ax3.set_title('Inference Speed', fontweight='bold')
ax3.grid(axis='x', alpha=0.3)
for i, (bar, val) in enumerate(zip(bars, model_comparison_df['FPS'])):
    ax3.text(val + 1, i, f'{val:.1f}', va='center', fontweight='bold')

# 4. Precision vs Recall
ax4 = axes[1, 0]
ax4.scatter(model_comparison_df['Recall'], model_comparison_df['Precision'],
           s=300, alpha=0.6, c=['#FF6B6B', '#4ECDC4', '#45B7D1'])
ax4.set_xlabel('Recall', fontweight='bold')
ax4.set_ylabel('Precision', fontweight='bold')
ax4.set_title('Precision-Recall Trade-off', fontweight='bold')
ax4.grid(alpha=0.3)
for idx, row in model_comparison_df.iterrows():
    ax4.annotate(row['Model'], (row['Recall'], row['Precision']),
                fontsize=10, ha='center', xytext=(0, 10), textcoords='offset points')

# 5. Parameter Count vs Accuracy
ax5 = axes[1, 1]
params_numeric = [float(p.replace('M', '')) for p in model_comparison_df['Parameters']]
ax5.plot(params_numeric, model_comparison_df['mAP@0.5:0.95'], 
         marker='o', linewidth=2, markersize=10, color='#4ECDC4')
ax5.set_xlabel('Parameters (Millions)', fontweight='bold')
ax5.set_ylabel('mAP@0.5:0.95', fontweight='bold')
ax5.set_title('Model Complexity vs Accuracy', fontweight='bold')
ax5.grid(alpha=0.3)

# 6. Overall Score (weighted)
ax6 = axes[1, 2]
# Normalize metrics to [0, 1] and create composite score
norm_map = model_comparison_df['mAP@0.5:0.95'] / model_comparison_df['mAP@0.5:0.95'].max()
norm_speed = 1 - (model_comparison_df['Inference Time (ms)'] / model_comparison_df['Inference Time (ms)'].max())
overall_score = 0.7 * norm_map + 0.3 * norm_speed

bars = ax6.bar(model_comparison_df['Model'], overall_score,
              color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
ax6.set_ylabel('Overall Score', fontweight='bold')
ax6.set_title('Composite Score (70% Accuracy + 30% Speed)', fontweight='bold')
ax6.grid(axis='y', alpha=0.3)
for i, (bar, val) in enumerate(zip(bars, overall_score)):
    ax6.text(i, val + 0.02, f'{val:.3f}', ha='center', fontweight='bold')

plt.suptitle('YOLOv8 Model Size Comparison - Blood Cell Detection', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('blood_cell_experiments/model_size_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 Model size analysis visualization saved!")

# Best model recommendation
print("\n" + "="*70)
print("MODEL SELECTION RECOMMENDATION")
print("="*70)

best_accuracy = model_comparison_df.loc[model_comparison_df['mAP@0.5:0.95'].idxmax()]
best_speed = model_comparison_df.loc[model_comparison_df['FPS'].idxmax()]
best_balanced = model_comparison_df.iloc[overall_score.argmax()]

print(f"\n🏆 Best Accuracy: {best_accuracy['Model']}")
print(f"   mAP@0.5:0.95: {best_accuracy['mAP@0.5:0.95']}")
print(f"   Use case: When accuracy is critical (research, diagnosis)")

print(f"\n⚡ Fastest: {best_speed['Model']}")
print(f"   FPS: {best_speed['FPS']}")
print(f"   Use case: Real-time applications, resource-constrained devices")

print(f"\n⚖️  Best Balanced: {best_balanced['Model']}")
print(f"   Overall Score: {overall_score.max():.3f}")
print(f"   Use case: Production deployment with good accuracy and speed")

In [ ]:
print("💾 CHECKING SAVED FILES")
print("="*70)

# All your results are saved to disk, not just in memory
saved_files = [
    "blood_cell_experiments/*/weights/best.pt",  # Model weights
    "blood_cell_experiments/results_comparison.csv",  # Results
    "blood_cell_experiments/*.png",  # Visualizations
    "blood_cell_dataset/",  # Dataset
]

print("\n✅ These files persist even if you close laptop or stop kernel:")
for pattern in saved_files:
    print(f"   • {pattern}")

print("\n❌ These are lost when kernel stops:")
lost_items = [
    "Variables in memory (all_results, comparison_df, etc.)",
    "Plots displayed but not saved",
    "Temporary variables",
]
for item in lost_items:
    print(f"   • {item}")

print("\n💡 Pro Tip: Always save results to files before closing!")